# CIFAR10 Image Classification

In [ ]:
import matplotlib.pylab as plt
import numpy as np
import tensorflow as tf

## 1. Load your data

In [ ]:
# Loading the data using the Keras function
(X_dev, Y_dev), (X_test, Y_test) = tf.keras.datasets.cifar10.load_data()  # The data comes already split in development and test sets

print(X_dev.shape)
print(X_test.shape)

## 2. Explore your data 

In [ ]:
#Diplaying the labels and the numbers of each label in development set
# Exploring the Data
print("Development set")
print("Images: ",X_dev.shape)
print("Labels shape:",Y_dev.shape)
print("\nNumber of classes:",np.unique(Y_dev).size)
print("\nClasses:",np.unique(Y_dev))
print("\nTest set")
print("Images: ",X_test.shape)
print("Labels shape: ",Y_test.shape)

The pixel size of image is 32X32 and they are RGB images(3 channels). Number of classes are 10. In development set we have 50,000 images and test set has 10,000 images.

In [ ]:
#Plotting Labels and Label Frequency
#The number of classes across samples in development set looks balanced
plt.figure()
plt.hist(Y_dev, bins = range(11))
plt.xlabel("Labels")
plt.ylabel("Label Frequency")
plt.show()

The number of classes across samples are balanced

In [ ]:
#Displaying some samples from the development set
sample_indexes = np.random.choice(np.arange(X_dev.shape[0], dtype = int),size = 30, replace = False)
plt.figure(figsize = (24,18))
for (ii,jj) in enumerate(sample_indexes):
    plt.subplot(5,6,ii+1)
    plt.imshow(X_dev[jj], cmap = "gray")
    plt.title("Label: %d" %Y_dev[jj])
plt.show()

In [ ]:
# Shuffle the samples in development set and split them into train set and validation set
indexes = np.arange(X_dev.shape[0], dtype = int)
np.random.shuffle(indexes)
X_dev = X_dev[indexes]
Y_dev = Y_dev[indexes]

nsplit = int(0.75*X_dev.shape[0]) # train/validation split, 75% for train, 25% for validation

# Train and validation split
X_train = X_dev[:nsplit]
Y_train = Y_dev[:nsplit]
X_val = X_dev[nsplit:]
Y_val = Y_dev[nsplit:]

print("\nTrain set")
print("Images: ",X_train.shape)
print("Labels shape: ",Y_train.shape)
print("\nValidation set")
print("Images: ",X_val.shape)
print("Labels shape: ",Y_val.shape)

In [ ]:
print(X_train.min(),X_train.max(),X_train.mean(),X_train.std())
print(X_val.min(),X_val.max(),X_val.mean(),X_val.std())

The statistical parameters of both test and validation sets are similar so, the split was fine.

## 3. Represent your labels using one hot encoding

In [ ]:
# Representing the labels using one hot encoding
Y_train_oh = tf.keras.utils.to_categorical(Y_train) 
Y_val_oh = tf.keras.utils.to_categorical(Y_val)
Y_test_oh = tf.keras.utils.to_categorical(Y_test)

print("Labels for train set:")
print(Y_train[:5])
print()
print("One hot encoded labels for train set:")
print(Y_train_oh[:5])

## 4. Data scaling and Data augmentation

In [ ]:
# Experimenting with different data scaling methods
norm_type = 1     

if norm_type == 0:            # norm_type = 0 -> min-max normalization

    X_train = X_train/255
    X_val = X_val/255
    X_test = X_test/255

elif norm_type == 1:          # norm_type = 1 - > standardization

    train_mean, train_std = X_train.mean(),X_train.std() 
    X_train = (X_train - train_mean)/train_std
    X_val = (X_val - train_mean)/train_std
    X_test = (X_test - train_mean)/train_std
else:
    pass

# Data Augumentation 

batch_size=32
gen_params = {"featurewise_center":False,"samplewise_center":False,"featurewise_std_normalization":False,\
              "samplewise_std_normalization":False,"zca_whitening":False,"rotation_range":20,"width_shift_range":0.1,"height_shift_range":0.1,\
              "shear_range":0.2, "zoom_range":0.1,"horizontal_flip":True,"fill_mode":'constant',\
               "cval": 0}
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(**gen_params)      
val_gen = tf.keras.preprocessing.image.ImageDataGenerator(**gen_params)

train_gen.fit(X_train,seed = 1)
val_gen.fit(X_val, seed = 1)

train_flow = train_gen.flow(X_train,Y_train_oh,batch_size = batch_size)
val_flow = val_gen.flow(X_val,Y_val_oh,batch_size = batch_size)

## Fully Connected Model

## 5. Define your  model, cost function, optimizer, learning rate

In [ ]:
# Fully Connected Neural Network Model

def my_model_fcn(ishape= (32,32,3),ndim=10,lr=1e-4):
  input_img = tf.keras.layers.Input(shape=ishape)
  flat      = tf.keras.layers.Flatten()(input_img)

  l1        = tf.keras.layers.Dense(1000, activation = 'relu')(flat)
  l2        = tf.keras.layers.Dense(750, activation = 'relu')(l1)
  drop1     = tf.keras.layers.Dropout(0.6)(l2)

  l3        = tf.keras.layers.Dense(750, activation = 'relu')(drop1)
  l4        = tf.keras.layers.Dense(500, activation = 'relu')(l3)
  drop2     = tf.keras.layers.Dropout(0.6)(l4)

  l5        = tf.keras.layers.Dense(500, activation = 'relu')(drop2)
  l6        = tf.keras.layers.Dense(250, activation = 'relu')(l5)
  out       = tf.keras.layers.Dense(ndim, activation='softmax')(l6)

  model     = tf.keras.models.Model(inputs= input_img, outputs=out)
  model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr), loss = 'categorical_crossentropy', metrics = ['accuracy'])
  return model

In [ ]:
# Model Summary 
model = my_model_fcn()
print(model.summary()) 

## 6. Define your callbacks (save your model, patience, etc.)

In [ ]:
# Defining callbacks

model_name_fcn = "team_(ENEL 645 L01-1)_FCN.h5"

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience = 20)

model_check_point = tf.keras.callbacks.ModelCheckpoint(model_name_fcn, monitor='val_loss', save_best_only=True, save_weights_only=True)

# Learning Rate Schedule 

def scheduler(epoch, lr):
    if epoch%50==0:
      lr = lr/1.5
    return lr

lr_schedule = tf.keras.callbacks.LearningRateScheduler(scheduler,verbose = 0)


## 7. Train your model

In [ ]:
# Training the Fully Connected Neural Network Model by using Data Augumentation
history = model.fit(train_flow, batch_size=64, epochs=120, callbacks=[model_check_point,early_stop,lr_schedule], validation_data=(val_flow))

## 8. Test your model

In [ ]:
# Testing the Fully Connected Neural Network Model

model.load_weights(model_name_fcn)            #loading weights of the best model
metrics = model.evaluate(X_test,Y_test_oh)

Ypred = model.predict(X_test).argmax(axis = 1)
wrong_indexes = np.where(Ypred != Y_test[:,0])[0]
print(wrong_indexes.size)

# Disaplying some samples from the development set

sample_indexes = np.random.choice(np.arange(wrong_indexes.shape[0], dtype = int),size = 30, replace = False)
plt.figure(figsize = (24,18))
for (ii,jj) in enumerate(sample_indexes):
    plt.subplot(5,6,ii+1)
    plt.imshow(X_test[wrong_indexes[jj]], cmap = "gray")
    plt.title("Label: %d, predicted: %d" %(Y_test[wrong_indexes[jj]],Ypred[wrong_indexes[jj]]))
plt.show()

In [ ]:
# Saving the weights of the best fully connected neural network model into a .h5 file

model.save('/content/drive/MyDrive/Assignment 2 Weights/team_(ENEL 645 L01-1)_FCN.h5')

In [ ]:
# Train and Validation Loss Graph
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()
plt.grid()
plt.show()

# Train and Validation Accuracy Graph
plt.plot(history.history['accuracy'], label='Train accuracy')
plt.plot(history.history['val_accuracy'], label='Val accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()
plt.grid()
plt.show()

**Obseravations during Fully Connected Neural Network Model Building:**

*   Adding additional dense layers with more neurons did not improve the accuracy.
*   Overfitting of the Fully Connected Neural Network Model reduced significantly by adding the dropouts.
*   A Test Accuracy of 53% was achieved without using Data Augumentaion.

*   By using Data Augumentation the test accuracy of the Fully Connected Neural Network Model increased to 58.60%



## Convolutional Model

## 5. Define your  model, cost function, optimizer, learning rate

In [ ]:
# Convolutional Neural Network Model

def my_model_cnn(ishape = (32,32,3), k = 10, lr = 1e-4):
    model_input = tf.keras.layers.Input(shape = ishape)

    e1 = tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(model_input)
    e2 = tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(e1)
    e2_drop = tf.keras.layers.Dropout(0.25)(e2)

    l1 = tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu')(e2_drop)
    l2 = tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu')(l1)
    l2_drop = tf.keras.layers.Dropout(0.25)(l2)
    l3 = tf.keras.layers.MaxPool2D((2,2))(l2_drop)

    l4 = tf.keras.layers.Conv2D(256, (3,3), padding='same', activation='relu')(l3)
    l5 = tf.keras.layers.Conv2D(256, (3,3), padding='same', activation='relu')(l4)
    l5_drop = tf.keras.layers.Dropout(0.25)(l5)
    l6 = tf.keras.layers.MaxPool2D((2,2))(l5_drop)

    l7 = tf.keras.layers.Conv2D(512, (3,3), padding='same', activation='relu')(l6)
    l8 = tf.keras.layers.Conv2D(512, (3,3), padding='same', activation='relu')(l7)
    l8_drop = tf.keras.layers.Dropout(0.25)(l8)
    l9 = tf.keras.layers.MaxPool2D((2,2))(l8_drop)

    L10 = tf.keras.layers.Conv2D(512, (3,3), padding='same', activation='relu')(l9)
    L10_drop = tf.keras.layers.Dropout(0.25)(L10)
    L11 = tf.keras.layers.MaxPool2D((2,2))(L10_drop)

    flat = tf.keras.layers.Flatten()(L11)
    out = tf.keras.layers.Dense(k,activation = 'softmax')(flat)

    model2 = tf.keras.models.Model(inputs = model_input, outputs = out)
    model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='categorical_crossentropy', metrics = ["accuracy"])
    return model2

In [ ]:
# Model 2 Summary
model2 = my_model_cnn()
print(model2.summary())

## 6. Define your callbacks (save your model, patience, etc.)

In [ ]:
# Defining Callbacks
model_name_cnn = "team_(ENEL 645 L01-1)_CNN.h5"

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience = 20)

monitor = tf.keras.callbacks.ModelCheckpoint(model_name_cnn, monitor='val_loss',\
                                             verbose=0,save_best_only=True,\
                                             save_weights_only=True,\
                                             mode='min')

# Learning rate schedule
def scheduler(epoch, lr):
    if epoch%10 == 0:
        lr = lr/2
    return lr

lr_schedule = tf.keras.callbacks.LearningRateScheduler(scheduler,verbose = 0)



## 7. Train your model

In [ ]:
# Training the model using Data Augumentation
history2 = model2.fit(train_flow,epochs = 35, verbose = 1, callbacks= [early_stop, monitor, lr_schedule], validation_data = (val_flow))

## 8. Test your model

In [ ]:
# Testing the Convolutional Neural Network Model

model2.load_weights(model_name_cnn)
metrics = model2.evaluate(X_test,Y_test_oh)

Ypred = model2.predict(X_test).argmax(axis = 1)
wrong_indexes = np.where(Ypred != Y_test[:,0])[0]
print(wrong_indexes.size)

# Disaplying some samples from the development set
sample_indexes = np.random.choice(np.arange(wrong_indexes.shape[0], dtype = int),size = 30, replace = False)
plt.figure(figsize = (24,18))
for (ii,jj) in enumerate(sample_indexes):
    plt.subplot(5,6,ii+1)
    plt.imshow(X_test[wrong_indexes[jj]], cmap = "gray")
    plt.title("Label: %d, predicted: %d" %(Y_test[wrong_indexes[jj]],Ypred[wrong_indexes[jj]]))
plt.show()

In [ ]:
# Saving the weights of the best Convolutional Neural Network into a .h5 file

model2.save('/content/drive/MyDrive/Assignment 2 Weights/team_(ENEL 645 L01-1)_CNN.h5')

In [ ]:
# Train and Validation Loss Graph
plt.plot(history2.history['loss'], label='Train loss')
plt.plot(history2.history['val_loss'], label='Val loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()
plt.grid()
plt.show()

# Train and Validation Accuracy Graph
plt.plot(history2.history['accuracy'], label='Train accuracy')
plt.plot(history2.history['val_accuracy'], label='Val accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()
plt.grid()
plt.show()

**Observations during Convolutional Neural Network Model Building:**


*   Adding additional dense layer after flattening did not improve accuracy.
*   Current model makes use of 9 convolution layers. Adding additional convolutional layer did not help improve the model accuracy.
*   A Test Accuracy of 75% was achieved without using Data Augumentaion.

*   By using Data Augumentation the test accuracy of the Convolutional Neural Network Model increased to 83.27%


